# Hierarchy-Agnostic Text Classification — reproducible pipeline

**Manuscript:** ARRAY-D-26-01504, *Beyond Predefined Trees: Hierarchy-Agnostic
Representation Learning for Text Classification*

This notebook computes every reported result from the data. No metric is
hard-coded anywhere: each figure and table is derived from the JSON written by
the experiment runners, so a plot cannot drift from the number it depicts.

**Outputs** are written to three folders:

| Folder | Contents |
|---|---|
| `Figures/` | publication figures, PNG at 300 dpi and vector PDF |
| `Tables/`  | every reported table as CSV and Markdown |
| `Others/`  | configuration, environment, raw per-seed JSON, label structure, findings |

**What the pipeline establishes.** Consistency-constrained decoding raises
hierarchical F1 significantly and removes invalid label paths entirely; the
hierarchy-discovery module contributes nothing measurable; and dynamic cluster
induction detects unseen categories at chance. The last two are negative
results, and they are reported as such.

---
*Google Colab version.* Outputs are written to
`MyDrive/Outputs/HTC/{Figures,Tables,Others}`.

Unlike the local environment, Colab can reach the model hub and provides a GPU,
so the transformer representation the manuscript describes can actually be used
here. Set `ENCODER = 'transformer'` below to do so. Runtime → Change runtime
type → GPU is recommended for that path.


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Configuration
OUT_ROOT   = '/content/drive/MyDrive/Outputs/HTC'   #@param {type:"string"}
DRIVE_ZIP  = '/content/drive/MyDrive/archive.zip'   #@param {type:"string"}
ENCODER    = 'tfidf_svd'                            #@param ["tfidf_svd", "transformer"]
MODEL_NAME = 'distilbert-base-uncased'              #@param {type:"string"}
SEEDS      = [0, 1, 2, 3, 4]                        #@param
MAX_LENGTH = 192                                    #@param {type:"integer"}

import os, json, zipfile, shutil
for sub in ('Figures', 'Tables', 'Others'):
    os.makedirs(os.path.join(OUT_ROOT, sub), exist_ok=True)
print('outputs ->', OUT_ROOT)

In [ ]:
#@title Dependencies
!pip -q install scikit-learn pandas matplotlib scipy tabulate
if ENCODER == 'transformer':
    !pip -q install -U transformers
    import torch
    print('GPU available:', torch.cuda.is_available(),
          '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
os.environ['HTC_SEEDS'] = ','.join(str(s) for s in SEEDS)
print('ready; seeds =', os.environ['HTC_SEEDS'])

In [ ]:
#@title Stage the dataset
# The Kaggle archive (train_40k.csv, val_10k.csv, unlabeled_150k.csv) is expected
# in Drive. Source:
#   https://www.kaggle.com/datasets/kashnitsky/hierarchical-text-classification
import os, zipfile
os.makedirs('data', exist_ok=True)
if os.path.exists(DRIVE_ZIP):
    with zipfile.ZipFile(DRIVE_ZIP) as z:
        z.extractall('data')
    # flatten any nested folder produced by the archive
    for root, _, files in os.walk('data'):
        for f in files:
            if f.endswith('.csv') and root != 'data':
                os.replace(os.path.join(root, f), os.path.join('data', f))
print(sorted(f for f in os.listdir('data') if f.endswith('.csv')))
assert os.path.exists('data/train_40k.csv'), 'train_40k.csv not found - check DRIVE_ZIP'

## Pipeline modules

In [ ]:
%%writefile htc_data.py
"""
Data preparation and evaluation protocol for the hierarchy-agnostic HTC study.

Corpus: Kaggle "Hierarchical Text Classification" (Amazon product reviews),
files train_40k.csv, val_10k.csv, unlabeled_150k.csv.

Two evaluation protocols are defined:

  P1 "published"        - train on train_40k, test on val_10k, as distributed.
                          NOTE: 60.2% of val_10k rows share a productId with
                          train_40k, so this protocol is optimistic.
  P2 "product-disjoint" - the 50k labelled rows are pooled and split by
                          productId, so no product appears in more than one
                          partition. This is the protocol we report as primary.

Input text is the review body (Text) only. The Title column holds the product
name, which is an almost perfect surrogate for the product category; including
it inflates every model and measures retrieval of the product name rather than
classification of the review. It is excluded and this is reported.
"""
import hashlib
import numpy as np
import pandas as pd

DATA = "data/"
LEVELS = ["Cat1", "Cat2", "Cat3"]
UNK = "unknown"


def load_labelled(drop_dupes=True):
    tr = pd.read_csv(DATA + "train_40k.csv")
    va = pd.read_csv(DATA + "val_10k.csv")
    tr["origin"] = "train_40k"
    va["origin"] = "val_10k"
    df = pd.concat([tr, va], ignore_index=True)
    df["Text"] = df["Text"].astype(str).str.strip()
    df = df[df["Text"].str.len() >= 20].reset_index(drop=True)
    if drop_dupes:
        df = df.drop_duplicates(subset=["Text"]).reset_index(drop=True)
    for c in LEVELS:
        df[c] = df[c].astype(str).str.strip()
    df["path2"] = df.Cat1 + " > " + df.Cat2
    df["path3"] = df.Cat1 + " > " + df.Cat2 + " > " + df.Cat3
    return df


def reference_hierarchy(df):
    """Parent maps induced by the reference labels."""
    p2 = df.groupby("Cat2")["Cat1"].agg(lambda s: s.value_counts().index[0]).to_dict()
    multi2 = df.groupby("Cat2")["Cat1"].nunique()
    multi3 = df[df.Cat3 != UNK].groupby("Cat3")["Cat2"].nunique()
    return {
        "parent_of_cat2": p2,
        "cat2_multi_parent": int((multi2 > 1).sum()),
        "cat3_multi_parent": int((multi3 > 1).sum()),
        "n_cat3_labelled": int(multi3.shape[0]),
    }


def _grp_hash(pid, salt):
    return int(hashlib.md5((str(pid) + "|" + str(salt)).encode()).hexdigest()[:8], 16) / 0xFFFFFFFF


def split_product_disjoint(df, seed=0, frac=(0.70, 0.10, 0.20)):
    """Group split by productId. Deterministic given seed."""
    h = df.productId.map(lambda p: _grp_hash(p, seed))
    a, b = frac[0], frac[0] + frac[1]
    part = np.where(h < a, "train", np.where(h < b, "val", "test"))
    return pd.Series(part, index=df.index)


def split_published(df):
    return pd.Series(np.where(df.origin == "train_40k", "train", "test"), index=df.index)


# ----------------------------------------------------------------- metrics
def ancestor_set(c1, c2, c3):
    s = {("1", c1), ("2", c1 + ">" + c2)}
    if c3 != UNK:
        s.add(("3", c1 + ">" + c2 + ">" + c3))
    return s


def hierarchical_prf(true_rows, pred_rows):
    """Standard hierarchical P/R/F1 over ancestor sets (Kiritchenko et al.)."""
    inter = tp = fp = fn = 0
    for t, p in zip(true_rows, pred_rows):
        ts, ps = ancestor_set(*t), ancestor_set(*p)
        inter = len(ts & ps)
        tp += inter
        fp += len(ps) - inter
        fn += len(ts) - inter
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    return P, R, F


def invalid_path_rate(pred_rows, valid_edges2, valid_edges3):
    """Fraction of predicted paths that are not present in the reference DAG."""
    bad = 0
    for c1, c2, c3 in pred_rows:
        ok = (c1, c2) in valid_edges2
        if ok and c3 != UNK:
            ok = (c2, c3) in valid_edges3
        if not ok:
            bad += 1
    return bad / len(pred_rows)


def valid_edge_sets(df):
    e2 = set(zip(df.Cat1, df.Cat2))
    e3 = set(zip(df[df.Cat3 != UNK].Cat2, df[df.Cat3 != UNK].Cat3))
    return e2, e3


In [ ]:
%%writefile baselines.py
"""Reference baselines: flat, per-level, and top-down predefined-tree classifiers."""
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
import htc_data as H

UNK = H.UNK


def vectoriser(seed=0):
    return TfidfVectorizer(sublinear_tf=True, min_df=3, max_df=0.6,
                           ngram_range=(1, 2), strip_accents="unicode",
                           lowercase=True, max_features=300000)


def _clf(seed):
    return LinearSVC(C=0.5, random_state=seed, max_iter=3000)


class FlatLeaf:
    """One classifier over the complete label path (structure ignored)."""
    name = "Flat classifier (full-path target)"

    def __init__(self, seed=0):
        self.seed = seed

    def fit(self, X, df):
        self.m = _clf(self.seed).fit(X, df.path3.values)
        return self

    def predict(self, X):
        return [tuple(p.split(" > ")) for p in self.m.predict(X)]


class PerLevel:
    """Three independent classifiers, one per level; no structure enforced."""
    name = "Per-level independent classifiers"

    def __init__(self, seed=0):
        self.seed = seed

    def fit(self, X, df):
        self.ms = [_clf(self.seed).fit(X, df[c].values) for c in H.LEVELS]
        return self

    def predict(self, X):
        cols = [m.predict(X) for m in self.ms]
        return list(zip(*cols))


class TopDownTree:
    """Predefined-hierarchy baseline: a local classifier per parent node."""
    name = "Top-down classifier chain (predefined tree)"

    def __init__(self, seed=0, min_train=8):
        self.seed = seed
        self.min_train = min_train

    def _fit_node(self, X, y, idx):
        ys = y[idx]
        uniq = np.unique(ys)
        if len(uniq) == 1 or len(idx) < self.min_train:
            return ("const", uniq[0] if len(uniq) else UNK)
        return ("model", _clf(self.seed).fit(X[idx], ys))

    def fit(self, X, df):
        c1, c2, c3 = (df[c].values for c in H.LEVELS)
        self.root = _clf(self.seed).fit(X, c1)
        self.n2, self.n3 = {}, {}
        for p in np.unique(c1):
            idx = np.where(c1 == p)[0]
            self.n2[p] = self._fit_node(X, c2, idx)
        for p in np.unique(c2):
            idx = np.where(c2 == p)[0]
            self.n3[p] = self._fit_node(X, c3, idx)
        return self

    def _apply(self, node, X, rows):
        kind, obj = node
        if kind == "const":
            return np.array([obj] * len(rows))
        return obj.predict(X[rows])

    def predict(self, X):
        n = X.shape[0]
        p1 = self.root.predict(X)
        p2 = np.empty(n, dtype=object)
        for p in np.unique(p1):
            rows = np.where(p1 == p)[0]
            p2[rows] = self._apply(self.n2[p], X, rows)
        p3 = np.empty(n, dtype=object)
        for p in np.unique(p2):
            node = self.n3.get(p, ("const", UNK))
            rows = np.where(p2 == p)[0]
            p3[rows] = self._apply(node, X, rows)
        return list(zip(p1, p2, p3))


In [ ]:
%%writefile embeddings.py
"""
Pluggable document representations.

Two encoders are provided behind one interface so that the rest of the pipeline
is unchanged when the representation changes:

    tfidf_svd    TF-IDF -> TruncatedSVD(256) -> L2. No pretrained weights, runs
                 anywhere, but a weak encoder (explained variance ~0.12).
    transformer  Mean-pooled (or CLS) hidden states of a pretrained checkpoint,
                 L2-normalised. Requires network access to the model hub and,
                 in practice, a GPU.

Both return an L2-normalised dense matrix suitable for cosine clustering.
"""
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize


class TfidfSvdEncoder:
    kind = 'tfidf_svd'

    def __init__(self, dim=256, seed=0):
        self.dim, self.seed = dim, seed

    def fit(self, Xtf, texts=None):
        self.svd = TruncatedSVD(n_components=self.dim, random_state=self.seed).fit(Xtf)
        self.explained = float(self.svd.explained_variance_ratio_.sum())
        return self

    def transform(self, Xtf, texts=None):
        return normalize(self.svd.transform(Xtf))

    def describe(self):
        return {'encoder': self.kind, 'dim': self.dim,
                'explained_variance': round(getattr(self, 'explained', float('nan')), 4)}


class TransformerEncoder:
    """Frozen pretrained encoder. Nothing is fine-tuned: the representation is
    extracted once and reused, which keeps the comparison against the linear
    encoder a comparison of representations rather than of training budgets."""
    kind = 'transformer'

    def __init__(self, model_name='distilbert-base-uncased', max_length=192,
                 batch_size=64, pooling='mean', device=None, seed=0):
        self.model_name = model_name
        self.max_length = max_length
        self.batch_size = batch_size
        self.pooling = pooling
        self.device = device
        self.seed = seed
        self._m = None

    def _load(self):
        if self._m is not None:
            return
        import torch
        from transformers import AutoTokenizer, AutoModel
        self.torch = torch
        if self.device is None:
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.tok = AutoTokenizer.from_pretrained(self.model_name)
        self._m = AutoModel.from_pretrained(self.model_name).to(self.device).eval()

    def fit(self, Xtf=None, texts=None):
        self._load()
        return self

    def transform(self, Xtf=None, texts=None):
        assert texts is not None, 'TransformerEncoder needs raw texts'
        self._load()
        torch = self.torch
        out = []
        texts = list(map(str, texts))
        with torch.no_grad():
            for i in range(0, len(texts), self.batch_size):
                b = texts[i:i + self.batch_size]
                enc = self.tok(b, padding=True, truncation=True,
                               max_length=self.max_length, return_tensors='pt').to(self.device)
                h = self._m(**enc).last_hidden_state
                if self.pooling == 'cls':
                    v = h[:, 0]
                else:
                    mask = enc['attention_mask'].unsqueeze(-1).float()
                    v = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
                out.append(v.cpu().numpy())
        return normalize(np.vstack(out))

    def describe(self):
        return {'encoder': self.kind, 'model_name': self.model_name,
                'max_length': self.max_length, 'pooling': self.pooling,
                'fine_tuned': False}


def get_encoder(kind='tfidf_svd', **kw):
    return TransformerEncoder(**kw) if kind == 'transformer' else TfidfSvdEncoder(**kw)


In [ ]:
%%writefile repr_util.py
"""
Single point of control for the document representation used by every runner.

By default the linear TF-IDF/SVD encoder is fitted on the training partition.
If the environment variable HTC_EMB_CACHE points at a .npy file holding one row
per document of labelled.pkl (in that file's order), those precomputed
embeddings are used instead - this is how the Colab notebook substitutes a
transformer without touching any other code.
"""
import os
import numpy as np
from sklearn.preprocessing import normalize
import proposed as PR


def _cache():
    p = os.environ.get('HTC_EMB_CACHE', '')
    return p if p and os.path.exists(p) else None


def describe():
    c = _cache()
    return {'source': 'cached embedding matrix', 'path': c} if c else \
           {'source': 'TF-IDF + TruncatedSVD', 'dim': 256}


def build(Xtr, Xothers, tr, others, seed=0):
    """Return (Ztr, [Z for each of `others`]).

    Xothers / others are parallel lists so the same call serves the two-way
    (train/test) and three-way (train/val/test) cases.
    """
    c = _cache()
    if c:
        Z = np.load(c)
        Ztr = normalize(Z[tr.index.values])
        return Ztr, [normalize(Z[o.index.values]) for o in others]
    ue = PR.UnifiedEmbedding(256, seed).fit(Xtr)
    return ue.transform(Xtr), [ue.transform(X) for X in Xothers]


In [ ]:
%%writefile proposed.py
"""
Hierarchy-agnostic classification framework - a concrete, executable
implementation of the components described in Sections 3.3-3.7 of the
manuscript (Eqs. 6-19). Every free choice left unspecified in the manuscript is
fixed here explicitly and reported.

Components
----------
Unified embedding space (Eqs. 6-8)
    TF-IDF -> truncated SVD (d=256) -> L2 normalisation. This is a linear
    encoder, not a transformer; the transformer variant is supported through
    `embeddings=` but requires pretrained weights, which the execution
    environment cannot download (see report).

Emergent concept clustering (Eq. 9)
    Spherical k-means (MiniBatchKMeans on L2-normalised vectors, i.e. cosine
    geometry) at K clusters. K selected on the validation partition.

Dynamic cluster induction (Eq. 10)
    A document whose cosine similarity to its nearest centroid falls below
    tau = mu - 2*sigma of the training assignment distribution is declared
    out-of-cluster and seeds a new centroid. Used for the evolving-label-space
    experiment.

Evolving label graph (Eq. 11)
    Weighted bipartite co-occurrence between clusters and observed labels,
    w(c, l) = P(l | c). Cluster-to-cluster edges are induced by shared label
    mass with a threshold theta.

Multi-level classifier (Eqs. 14-15)
    Multinomial logistic regression per level over [unified embedding ||
    cluster posterior].

Confidence calibration (Eq. 17)
    Temperature scaling per level, fitted on the validation partition.

Hierarchical consistency + constraints (Eqs. 16, 18, 19)
    Exact joint decoding over the label lattice: the predicted path is
    argmax over paths admissible under the *discovered* edge set of
    log p1 + log p2 + log p3. This replaces the manuscript's unspecified
    "consistency module" with a decoding rule that is both defined and
    measurable.
"""
import numpy as np
from scipy.special import log_softmax
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import LinearSVC
from sklearn.preprocessing import normalize, LabelEncoder
from scipy import sparse as sp
import htc_data as H

UNK = H.UNK


class UnifiedEmbedding:
    def __init__(self, dim=256, seed=0):
        self.dim, self.seed = dim, seed

    def fit(self, Xtf):
        self.svd = TruncatedSVD(n_components=self.dim, random_state=self.seed)
        self.svd.fit(Xtf)
        self.explained = float(self.svd.explained_variance_ratio_.sum())
        return self

    def transform(self, Xtf):
        return normalize(self.svd.transform(Xtf))


class HierarchyAgnostic:
    name = "Proposed hierarchy-agnostic framework"

    def __init__(self, seed=0, n_clusters=300, theta=0.0,
                 use_clusters=True, use_consistency=True, use_calibration=True):
        self.seed = seed
        self.K = n_clusters
        self.theta = theta
        self.use_clusters = use_clusters
        self.use_consistency = use_consistency
        self.use_calibration = use_calibration

    # ---------------------------------------------------------- discovery
    def _cluster(self, Z):
        self.km = MiniBatchKMeans(n_clusters=self.K, random_state=self.seed,
                                  n_init=5, batch_size=2048, max_iter=200)
        a = self.km.fit_predict(Z)
        sim = (Z * self.km.cluster_centers_[a]).sum(1)
        self.tau = float(sim.mean() - 2 * sim.std())
        return a

    TOPC = 8  # cluster posterior is truncated to its TOPC largest entries

    def _cluster_feats(self, Z):
        """Sparse truncated cluster posterior. Retaining only the TOPC nearest
        centroids keeps the augmented design matrix sparse; the discarded mass
        is negligible at the softmax temperature used."""
        d = Z @ self.km.cluster_centers_.T
        p = np.exp(log_softmax(d * 8.0, axis=1))  # bounded in [0,1]; keeps the
        n, K = p.shape                            # design matrix well conditioned
        top = np.argpartition(-p, self.TOPC, axis=1)[:, :self.TOPC]
        rows = np.repeat(np.arange(n), self.TOPC)
        cols = top.ravel()
        vals = p[rows, cols]
        return sp.csr_matrix((vals, (rows, cols)), shape=(n, K))

    def _label_graph(self, assign, df):
        """P(label | cluster) at each level, and the discovered edge sets."""
        self.edges2, self.edges3 = set(), set()
        for lv, (a, b) in enumerate([("Cat1", "Cat2"), ("Cat2", "Cat3")]):
            pass
        # cluster -> label mass
        self.cl_label = {}
        for lv in H.LEVELS:
            m = {}
            for c, l in zip(assign, df[lv].values):
                m.setdefault(c, {}).setdefault(l, 0)
                m[c][l] += 1
            self.cl_label[lv] = m
        # discovered edges: pairs co-occurring within any cluster above theta
        tot = len(df)
        c1, c2, c3 = df.Cat1.values, df.Cat2.values, df.Cat3.values
        for c in np.unique(assign):
            idx = np.where(assign == c)[0]
            n = len(idx)
            if n == 0:
                continue
            for i in idx:
                self.edges2.add((c1[i], c2[i]))
                if c3[i] != UNK:
                    self.edges3.add((c2[i], c3[i]))
        return self

    # -------------------------------------------------------------- fit
    def fit(self, Ztr, df_tr, Zva=None, df_va=None, Xtr=None, Xva=None):
        self._Xtr = Xtr
        if self.use_clusters:
            assign = self._cluster(Ztr)
            self._label_graph(assign, df_tr)
            F = self._features(Ztr, Xtr)
        else:
            # discovery ablation: edges still needed for decoding, taken from data
            self.edges2 = set(zip(df_tr.Cat1, df_tr.Cat2))
            m = df_tr.Cat3 != UNK
            self.edges3 = set(zip(df_tr[m].Cat2, df_tr[m].Cat3))
            F = self._features(Ztr, Xtr)
        self.enc, self.clf = {}, {}
        for lv in H.LEVELS:
            e = LabelEncoder().fit(df_tr[lv].values)
            self.enc[lv] = e
            # One-vs-rest linear margins converted to calibrated posteriors by
            # temperature-scaled softmax (Eq. 17). Chosen over multinomial
            # logistic regression purely for tractability at 464 classes.
            self.clf[lv] = LinearSVC(C=0.5, random_state=self.seed,
                                     max_iter=3000).fit(F, e.transform(df_tr[lv].values))
        self.T = {lv: 1.0 for lv in H.LEVELS}
        if self.use_calibration and Zva is not None:
            self._calibrate(Zva, df_va, Xva)
        self._build_lattice()
        return self

    def _features(self, Z, X=None):
        """Sparse lexical features concatenated with the cluster posterior
        (Eq. 11 evidence). The dense unified embedding drives clustering; the
        sparse features drive discrimination."""
        if X is None:
            raise ValueError("sparse features required")
        if not self.use_clusters:
            return X
        return sp.hstack([X, self._cluster_feats(Z)], format="csr")

    def _calibrate(self, Zva, df_va, Xva=None):
        F = self._features(Zva, Xva)
        for lv in H.LEVELS:
            logit = self.clf[lv].decision_function(F)
            if logit.ndim == 1:
                logit = np.vstack([-logit, logit]).T
            # validation rows carrying a label unseen in training cannot be
            # scored under this level's label set and are excluded from the
            # temperature fit (they remain in the test evaluation as errors)
            known = np.isin(df_va[lv].values, self.enc[lv].classes_)
            if known.sum() < 50:
                continue
            logit = logit[known]
            y = self.enc[lv].transform(df_va[lv].values[known])
            best, bt = np.inf, 1.0
            for t in np.linspace(0.4, 4.0, 19):
                lp = log_softmax(logit / t, axis=1)
                nll = -lp[np.arange(len(y)), y].mean()
                if nll < best:
                    best, bt = nll, t
            self.T[lv] = float(bt)

    def _build_lattice(self):
        """Admissible (l1,l2,l3) paths under the discovered edge set."""
        i1 = {l: i for i, l in enumerate(self.enc["Cat1"].classes_)}
        i2 = {l: i for i, l in enumerate(self.enc["Cat2"].classes_)}
        i3 = {l: i for i, l in enumerate(self.enc["Cat3"].classes_)}
        paths = []
        child3 = {}
        for a, b in self.edges3:
            child3.setdefault(a, []).append(b)
        for a, b in self.edges2:
            if a not in i1 or b not in i2:
                continue
            kids = child3.get(b, [])
            opts = [c for c in kids if c in i3] + ([UNK] if UNK in i3 else [])
            for c in opts:
                paths.append((i1[a], i2[b], i3[c], a, b, c))
        self.paths = paths
        self.P1 = np.array([p[0] for p in paths])
        self.P2 = np.array([p[1] for p in paths])
        self.P3 = np.array([p[2] for p in paths])
        self.Plab = [(p[3], p[4], p[5]) for p in paths]

    # ---------------------------------------------------------- predict
    def _logprobs(self, Z, X=None):
        F = self._features(Z, X)
        out = {}
        for lv in H.LEVELS:
            logit = self.clf[lv].decision_function(F)
            if logit.ndim == 1:
                logit = np.vstack([-logit, logit]).T
            out[lv] = log_softmax(logit / self.T[lv], axis=1)
        return out

    def predict(self, Z, X=None, batch=2000):
        lp = self._logprobs(Z, X)
        n = Z.shape[0]
        if not self.use_consistency:
            return [tuple(self.enc[lv].classes_[lp[lv][i].argmax()] for lv in H.LEVELS)
                    for i in range(n)]
        out = []
        for s in range(0, n, batch):
            e = min(s + batch, n)
            sc = (lp["Cat1"][s:e][:, self.P1]
                  + lp["Cat2"][s:e][:, self.P2]
                  + lp["Cat3"][s:e][:, self.P3])
            best = sc.argmax(1)
            out.extend(self.Plab[b] for b in best)
        return out


In [ ]:
%%writefile evaluate.py
"""Evaluation harness: flat and hierarchical metrics, shared by all models."""
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import htc_data as H

UNK = H.UNK


def score(true_rows, pred_rows, e2, e3):
    t = list(zip(*true_rows))
    p = list(zip(*pred_rows))
    out = {}
    for i, lv in enumerate(H.LEVELS):
        yt, yp = np.array(t[i]), np.array(p[i])
        out[f"acc_L{i+1}"] = accuracy_score(yt, yp)
        out[f"microF1_L{i+1}"] = f1_score(yt, yp, average="micro", zero_division=0)
        out[f"macroF1_L{i+1}"] = f1_score(yt, yp, average="macro", zero_division=0)
    # level 3 restricted to documents that genuinely have a level-3 label
    m = np.array(t[2]) != UNK
    if m.sum():
        out["acc_L3_labelled"] = accuracy_score(np.array(t[2])[m], np.array(p[2])[m])
        out["macroF1_L3_labelled"] = f1_score(np.array(t[2])[m], np.array(p[2])[m],
                                              average="macro", zero_division=0)
    out["exact_path"] = float(np.mean([a == b for a, b in zip(true_rows, pred_rows)]))
    hP, hR, hF = H.hierarchical_prf(true_rows, pred_rows)
    out["hP"], out["hR"], out["hF1"] = hP, hR, hF
    out["invalid_path_rate"] = H.invalid_path_rate(pred_rows, e2, e3)
    return out


def fmt(d, keys=None):
    keys = keys or ["acc_L1", "acc_L2", "acc_L3", "macroF1_L1", "macroF1_L2",
                    "macroF1_L3", "exact_path", "hP", "hR", "hF1", "invalid_path_rate"]
    return "  ".join(f"{k}={d[k]*100:.2f}" for k in keys if k in d)


In [ ]:
%%writefile outputs.py
"""
Output generation: writes every figure, table and auxiliary artefact of the
study into a three-folder structure.

    <root>/Figures/   publication figures, PNG at 300 dpi and PDF
    <root>/Tables/    every reported table as CSV and as Markdown
    <root>/Others/    configuration, environment, raw per-seed results, logs,
                      discovered structure, dataset inventory

Nothing here recomputes results; it consumes the JSON produced by the
experiment runners so that figures and tables can never drift from the numbers
that were actually measured.
"""
import json, os, platform, sys, datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

import htc_data as H

NAVY, TEAL, AMBER, GREY, RED, PLUM = '#1F3864', '#2E8B8B', '#C77D28', '#8C8C8C', '#A33131', '#7A5C8E'
PAL = [NAVY, TEAL, AMBER, RED, '#5B7C99', PLUM]

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': .25, 'grid.linewidth': .6,
                     'savefig.dpi': 300, 'figure.dpi': 120})

ORDER = ['Flat classifier (full-path target)',
         'Per-level independent classifiers',
         'Top-down classifier chain (predefined tree)',
         'Proposed (full)']
ABL = ['Proposed (full)', 'Proposed (-consistency)',
       'Proposed (-discovery)', 'Proposed (-calibration)']
SHORT = {'Flat classifier (full-path target)': 'Flat (full path)',
         'Per-level independent classifiers': 'Per-level independent',
         'Top-down classifier chain (predefined tree)': 'Top-down (predefined tree)',
         'Proposed (full)': 'Proposed (full)',
         'Proposed (-consistency)': 'Proposed − consistency',
         'Proposed (-discovery)': 'Proposed − discovery',
         'Proposed (-calibration)': 'Proposed − calibration'}


class Out:
    def __init__(self, root):
        self.root = root
        for s in ('Figures', 'Tables', 'Others'):
            os.makedirs(os.path.join(root, s), exist_ok=True)
        self.manifest = []

    def fig(self, f, name, title):
        for ext in ('png', 'pdf'):
            p = os.path.join(self.root, 'Figures', f'{name}.{ext}')
            f.savefig(p, bbox_inches='tight')
        plt.close(f)
        self.manifest.append(('Figures', f'{name}.png / .pdf', title))
        print(f'  Figures/{name}.png', flush=True)

    def table(self, df, name, title, index=False):
        p = os.path.join(self.root, 'Tables', f'{name}.csv')
        df.to_csv(p, index=index)
        with open(os.path.join(self.root, 'Tables', f'{name}.md'), 'w') as fh:
            fh.write(f'**{title}**\n\n' + df.to_markdown(index=index) + '\n')
        self.manifest.append(('Tables', f'{name}.csv / .md', title))
        print(f'  Tables/{name}.csv', flush=True)

    def other(self, name, content, title):
        p = os.path.join(self.root, 'Others', name)
        if isinstance(content, (dict, list)):
            json.dump(content, open(p, 'w'), indent=1, default=str)
        else:
            open(p, 'w').write(content)
        self.manifest.append(('Others', name, title))
        print(f'  Others/{name}', flush=True)


def _stat(R, model, key):
    d = R[model]
    return np.array([d[s][key] * 100 for s in sorted(d, key=int)])


def _pm(R, model, key, dec=2):
    v = _stat(R, model, key)
    return f'{v.mean():.{dec}f} ± {v.std(ddof=1):.{dec}f}'


def build(root='outputs', results_dir='.', df=None):
    o = Out(root)
    R = json.load(open(os.path.join(results_dir, 'results_seeds.json')))
    D = json.load(open(os.path.join(results_dir, 'results_discovery.json')))
    B = json.load(open(os.path.join(results_dir, 'results_baselines.json')))
    seeds = sorted(R[ORDER[0]], key=int)
    if df is None:
        df = H.load_labelled()

    print('Tables', flush=True)
    # --- T1 dataset inventory
    ref = H.reference_hierarchy(df)
    t1 = pd.DataFrame([
        ('Labelled documents (after cleaning)', len(df)),
        ('Mean review length (characters)', int(df.Text.str.len().mean())),
        ('Level-1 categories', df.Cat1.nunique()),
        ('Level-2 categories', df.Cat2.nunique()),
        ('Level-3 categories', df.Cat3.nunique()),
        ('Distinct label paths', df.groupby(['Cat1', 'Cat2', 'Cat3']).ngroups),
        ('Documents without a level-3 label', int((df.Cat3 == H.UNK).sum())),
        ('Level-2 categories with >1 parent', ref['cat2_multi_parent']),
        ('Level-3 categories with >1 parent', ref['cat3_multi_parent']),
        ('Largest level-1 class', int(df.Cat1.value_counts().iloc[0])),
        ('Smallest level-1 class', int(df.Cat1.value_counts().iloc[-1])),
    ], columns=['Property', 'Value'])
    o.table(t1, 'table1_dataset_statistics', 'Dataset inventory')

    # --- T2 main results
    KEY = [('acc_L1', 'Accuracy L1'), ('acc_L2', 'Accuracy L2'), ('acc_L3', 'Accuracy L3'),
           ('macroF1_L1', 'Macro-F1 L1'), ('macroF1_L2', 'Macro-F1 L2'),
           ('macroF1_L3', 'Macro-F1 L3'), ('exact_path', 'Exact path'),
           ('hP', 'Hier. precision'), ('hR', 'Hier. recall'), ('hF1', 'Hier. F1'),
           ('invalid_path_rate', 'Invalid paths')]
    t2 = pd.DataFrame([[SHORT[m]] + [_pm(R, m, k) for k, _ in KEY] for m in ORDER],
                      columns=['Model'] + [n for _, n in KEY])
    o.table(t2, 'table2_main_results',
            f'Main results, mean ± SD over {len(seeds)} seeds, product-disjoint split (%)')

    # --- T3 ablation + significance
    base = _stat(R, 'Proposed (full)', 'hF1')
    rows = []
    for m in ABL:
        v = _stat(R, m, 'hF1')
        if m == 'Proposed (full)':
            d, p = '—', '—'
        else:
            t, pv = stats.ttest_rel(base, v)
            d = f'{base.mean()-v.mean():+.2f}'
            p = '—' if np.isnan(pv) else (f'{pv:.1e}' if pv < 1e-3 else f'{pv:.3f}')
        rows.append([SHORT[m], _pm(R, m, 'hF1'), d, p, _pm(R, m, 'invalid_path_rate'),
                     _pm(R, m, 'acc_L1')])
    t3 = pd.DataFrame(rows, columns=['Configuration', 'Hier. F1', 'Δ vs full',
                                     'p (paired t)', 'Invalid paths', 'Accuracy L1'])
    o.table(t3, 'table3_ablation', 'Ablation with paired significance tests (%)')

    # --- T4 significance vs baselines
    rows = []
    for m in ORDER[:-1] + ABL[1:]:
        v = _stat(R, m, 'hF1')
        t, pv = stats.ttest_rel(base, v)
        rows.append([SHORT[m], f'{base.mean()-v.mean():+.2f}',
                     '—' if np.isnan(t) else f'{t:.2f}',
                     '—' if np.isnan(pv) else (f'{pv:.1e}' if pv < 1e-3 else f'{pv:.3f}'),
                     'yes' if (not np.isnan(pv) and pv < 0.05) else 'no'])
    t4 = pd.DataFrame(rows, columns=['Compared with', 'Δ Hier. F1', 't', 'p',
                                     'Significant at 0.05'])
    o.table(t4, 'table4_significance', 'Paired t-tests on hierarchical F1 across seeds')

    # --- T5 hierarchy recovery
    hr = D['hierarchy_recovery']
    rows = [[f'Discovered vs level {i+1}', hr[l]['K'], f"{hr[l]['ARI']:.3f}", f"{hr[l]['NMI']:.3f}"]
            for i, l in enumerate(['Cat1', 'Cat2', 'Cat3'])]
    rows.append(['Nested 64→6 vs level 1', 6,
                 f"{D['nested_coarse_vs_Cat1']['ARI']:.3f}",
                 f"{D['nested_coarse_vs_Cat1']['NMI']:.3f}"])
    t5 = pd.DataFrame(rows, columns=['Comparison', 'K', 'Adjusted Rand', 'Normalised MI'])
    o.table(t5, 'table5_hierarchy_recovery',
            'Agreement between the discovered partition and the reference hierarchy (0 = chance)')

    # --- T6 unseen categories
    ev = D['evolving']
    t6 = pd.DataFrame([
        ('Withheld level-2 categories', len(ev['held_out_categories'])),
        ('Test documents from withheld categories', ev['n_test_new']),
        ('Test documents from seen categories', ev['n_test_seen']),
        ('Mean similarity, seen', round(ev['mean_sim_seen'], 4)),
        ('Mean similarity, withheld', round(ev['mean_sim_new'], 4)),
        ('Induction threshold tau', round(ev['tau'], 4)),
        ('Detection rate at tau (%)', round(ev['detection_rate_new'] * 100, 2)),
        ('False-alarm rate at tau (%)', round(ev['false_alarm_rate_seen'] * 100, 2)),
        ('ROC AUC (chance = 0.500)', round(ev['roc_auc'], 3)),
    ], columns=['Quantity', 'Value'])
    o.table(t6, 'table6_unseen_categories', 'Detection of withheld categories')

    # --- T7 split comparison
    rows = []
    for m in ORDER[:-1]:
        p1, p2 = B.get(f'P1_published|{m}'), B.get(f'P2_product_disjoint|{m}')
        if not p1:
            continue
        rows.append([SHORT[m]] + [f'{p1[k]*100:.2f} / {p2[k]*100:.2f}'
                                  for k in ('acc_L1', 'acc_L2', 'acc_L3', 'hF1')])
    t7 = pd.DataFrame(rows, columns=['Model', 'Accuracy L1', 'Accuracy L2',
                                     'Accuracy L3', 'Hier. F1'])
    o.table(t7, 'table7_split_comparison',
            'Published split / product-disjoint split, single seed (%)')

    # --- T8 per-seed raw
    rows = []
    for m in R:
        for s in sorted(R[m], key=int):
            d = R[m][s]
            rows.append({'model': SHORT.get(m, m), 'seed': int(s),
                         **{k: round(d[k] * 100, 3) for k, _ in KEY}})
    t8 = pd.DataFrame(rows)
    o.table(t8, 'table8_per_seed_raw', 'Per-seed raw results (%)')

    print('Figures', flush=True)
    # --- F1 level-wise
    f, axes = plt.subplots(1, 2, figsize=(7.4, 2.9))
    for ax, met, name in [(axes[0], 'acc', 'Accuracy'), (axes[1], 'macroF1', 'Macro-F1')]:
        x = np.arange(3); w = .2
        for i, m in enumerate(ORDER):
            mu = [_stat(R, m, f'{met}_L{l}').mean() for l in (1, 2, 3)]
            sd = [_stat(R, m, f'{met}_L{l}').std(ddof=1) for l in (1, 2, 3)]
            ax.bar(x + (i - 1.5) * w, mu, w, yerr=sd, capsize=2, color=PAL[i],
                   label=SHORT[m], error_kw={'lw': .7})
        ax.set_xticks(x); ax.set_xticklabels(['Level 1', 'Level 2', 'Level 3'])
        ax.set_ylabel(f'{name} (%)'); ax.set_title(name, fontsize=9.5); ax.set_ylim(0, 100)
    axes[0].legend(fontsize=6.5, frameon=False, ncol=2, loc='upper right')
    o.fig(f, 'fig01_levelwise_performance',
          'Accuracy and macro-F1 by hierarchy level, mean ± SD over seeds')

    # --- F2 ablation
    f, axes = plt.subplots(1, 2, figsize=(7.4, 2.9))
    seq = ABL + ORDER[:-1]
    y = np.arange(len(seq))[::-1]
    cols = [NAVY] + [TEAL] * 3 + [GREY] * 3
    for ax, key, lab, xl in [(axes[0], 'hF1', 'Hierarchical F1 (%)', (55, 72)),
                             (axes[1], 'invalid_path_rate', 'Invalid label paths (%)', None)]:
        mu = [_stat(R, m, key).mean() for m in seq]
        sd = [_stat(R, m, key).std(ddof=1) for m in seq]
        ax.barh(y, mu, xerr=sd, color=cols, capsize=2, error_kw={'lw': .7})
        ax.set_yticks(y)
        ax.set_yticklabels([SHORT[m] for m in seq] if key == 'hF1' else [], fontsize=7.5)
        ax.set_xlabel(lab)
        if xl:
            ax.set_xlim(*xl)
    axes[0].set_title('Hierarchical F1', fontsize=9.5)
    axes[1].set_title('Structural validity', fontsize=9.5)
    o.fig(f, 'fig02_ablation', 'Ablation: hierarchical F1 and invalid-path rate')

    # --- F3 recovery
    f, ax = plt.subplots(figsize=(3.9, 2.8))
    lv = ['Cat1', 'Cat2', 'Cat3']; x = np.arange(3); w = .35
    ax.bar(x - w / 2, [hr[l]['ARI'] for l in lv], w, color=NAVY, label='Adjusted Rand index')
    ax.bar(x + w / 2, [hr[l]['NMI'] for l in lv], w, color=TEAL, label='Normalised MI')
    ax.set_xticks(x)
    ax.set_xticklabels([f"Level 1\n(K={hr['Cat1']['K']})", f"Level 2\n(K={hr['Cat2']['K']})",
                        f"Level 3\n(K={hr['Cat3']['K']})"])
    ax.set_ylim(0, 1); ax.set_ylabel('Agreement with reference')
    ax.legend(fontsize=7, frameon=False)
    ax.set_title('Recovery of the reference hierarchy', fontsize=9.5)
    o.fig(f, 'fig03_hierarchy_recovery',
          'Agreement between discovered partition and reference labels')

    # --- F4 unseen
    f, ax = plt.subplots(figsize=(3.9, 2.8))
    ax.bar([0, 1], [ev['mean_sim_seen'], ev['mean_sim_new']], .5, color=[NAVY, AMBER])
    ax.axhline(ev['tau'], color=RED, ls='--', lw=1,
               label=f"induction threshold $\\tau$ = {ev['tau']:.3f}")
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Seen categories', 'Withheld categories'])
    ax.set_ylabel('Mean similarity to nearest centroid')
    ax.set_title(f"Unseen-category detection (AUC = {ev['roc_auc']:.3f})", fontsize=9.5)
    ax.legend(fontsize=7, frameon=False, loc='lower right')
    o.fig(f, 'fig04_unseen_category_detection',
          'Nearest-centroid similarity for seen and withheld categories')

    # --- F5 label structure
    f, ax = plt.subplots(figsize=(7.4, 4.2))
    c1s = df.Cat1.value_counts(); order1 = list(c1s.index)
    xs1 = np.linspace(0, 1, len(order1))
    c2p = df.groupby('Cat2').Cat1.agg(lambda s: s.value_counts().index[0])
    c2c = df.Cat2.value_counts()
    grouped = {c: sorted([k for k, v in c2p.items() if v == c], key=lambda k: -c2c[k])
               for c in order1}
    xs2, col2 = {}, {}
    cur = 0; tot = sum(len(v) for v in grouped.values())
    for i, c in enumerate(order1):
        for k in grouped[c]:
            xs2[k] = cur / (tot - 1); col2[k] = PAL[i % len(PAL)]; cur += 1
    for i, c in enumerate(order1):
        for k in grouped[c]:
            ax.plot([xs1[i], xs2[k]], [1, 0], color=PAL[i % len(PAL)], lw=.6, alpha=.55)
        ax.scatter(xs1[i], 1, s=60 + 300 * c1s[c] / c1s.max(), color=PAL[i % len(PAL)], zorder=3)
        ax.text(xs1[i], 1.07, c, ha='center', fontsize=7.5)
    for k, x in xs2.items():
        ax.scatter(x, 0, s=8 + 120 * c2c[k] / c2c.max(), color=col2[k], zorder=3)
        ax.text(x, -0.06, k, ha='right', va='top', fontsize=5.2, rotation=60)
    ax.set_xlim(-.05, 1.05); ax.set_ylim(-.75, 1.2); ax.axis('off')
    ax.set_title(f'Reference label structure: {df.Cat1.nunique()} level-1 categories over '
                 f'{df.Cat2.nunique()} level-2 categories\n'
                 '(node area proportional to document frequency)', fontsize=9)
    o.fig(f, 'fig05_label_structure',
          'Reference hierarchy as a layered graph (replacement for the submitted Figure 20)')

    # --- F6 class imbalance
    f, axes = plt.subplots(1, 3, figsize=(7.4, 2.5))
    for ax, lv, ttl in zip(axes, H.LEVELS, ['Level 1', 'Level 2', 'Level 3']):
        v = df[lv].value_counts().values
        ax.plot(np.arange(1, len(v) + 1), v, color=NAVY, lw=1.2, marker='o' if len(v) < 10 else None,
                ms=3)
        if len(v) >= 10:
            ax.set_xscale('log')
        else:
            ax.set_xticks(np.arange(1, len(v) + 1))
        ax.set_yscale('log')
        ax.set_xlabel('Category rank'); ax.set_title(f'{ttl} ({len(v)} classes)', fontsize=9)
    axes[0].set_ylabel('Documents')
    f.suptitle('Class-frequency distribution by level (log-log)', fontsize=9.5, y=1.02)
    o.fig(f, 'fig06_class_distribution',
          'Long-tailed class distribution, the reason macro-F1 falls far below accuracy')

    # --- F7 split effect
    f, ax = plt.subplots(figsize=(4.6, 2.8))
    ms = [m for m in ORDER[:-1] if f'P1_published|{m}' in B]
    x = np.arange(len(ms)); w = .35
    ax.bar(x - w / 2, [B[f'P1_published|{m}']['hF1'] * 100 for m in ms], w,
           color=GREY, label='Published split')
    ax.bar(x + w / 2, [B[f'P2_product_disjoint|{m}']['hF1'] * 100 for m in ms], w,
           color=NAVY, label='Product-disjoint split')
    ax.set_xticks(x); ax.set_xticklabels([SHORT[m] for m in ms], fontsize=6.5)
    ax.set_ylabel('Hierarchical F1 (%)'); ax.set_ylim(50, 70)
    ax.legend(fontsize=7, frameon=False)
    ax.set_title('Effect of the evaluation split', fontsize=9.5)
    o.fig(f, 'fig07_split_effect', 'Published versus product-disjoint evaluation split')

    print('Others', flush=True)
    o.other('config.json', {
        'corpus': 'Kaggle Hierarchical Text Classification (Amazon reviews)',
        'input_field': 'Text (review body); Title excluded - it is the product name',
        'documents_after_cleaning': len(df),
        'protocol_primary': 'product-disjoint 70/10/20 by hashed productId',
        'protocol_secondary': 'published train_40k / val_10k split',
        'seeds': [int(s) for s in seeds],
        'tfidf': {'sublinear_tf': True, 'min_df': 3, 'max_df': 0.6,
                  'ngram_range': [1, 2], 'max_features': 300000},
        'unified_embedding': {'method': 'TruncatedSVD', 'dim': 256, 'l2_normalised': True},
        'clustering': {'method': 'MiniBatchKMeans', 'K': 300, 'n_init': 5, 'batch_size': 2048},
        'classifier': {'method': 'LinearSVC', 'C': 0.5, 'max_iter': 3000, 'scheme': 'one-vs-rest'},
        'calibration': {'method': 'temperature scaling', 'grid': [0.4, 4.0, 19],
                        'fitted_on': 'validation partition'},
        'decoding': 'argmax over admissible paths of log p1 + log p2 + log p3',
        'representation_caveat': ('Linear TF-IDF/SVD encoder, not a transformer. '
                                  'HuggingFace was unreachable in the environment where '
                                  'these results were produced.'),
    }, 'Full experimental configuration')

    o.other('environment.json', {
        'generated': datetime.datetime.now().isoformat(timespec='seconds'),
        'python': sys.version.split()[0], 'platform': platform.platform(),
        'numpy': np.__version__, 'pandas': pd.__version__,
        'matplotlib': matplotlib.__version__,
    }, 'Software environment')

    for src, dst, t in [('results_seeds.json', 'raw_results_per_seed.json', 'Raw per-seed metrics'),
                        ('results_discovery.json', 'raw_results_discovery.json', 'Raw discovery-experiment output'),
                        ('results_baselines.json', 'raw_results_split_comparison.json', 'Raw split-comparison output')]:
        p = os.path.join(results_dir, src)
        if os.path.exists(p):
            o.other(dst, json.load(open(p)), t)

    e2, e3 = H.valid_edge_sets(df)
    o.other('reference_edges.txt',
            'LEVEL 1 -> LEVEL 2\n' + '\n'.join(f'{a}\t{b}' for a, b in sorted(e2)) +
            '\n\nLEVEL 2 -> LEVEL 3\n' + '\n'.join(f'{a}\t{b}' for a, b in sorted(e3)),
            'Reference parent-child edge set')

    paths = (df.groupby(['Cat1', 'Cat2', 'Cat3']).size()
             .reset_index(name='documents').sort_values('documents', ascending=False))
    paths.to_csv(os.path.join(o.root, 'Others', 'label_paths.csv'), index=False)
    o.manifest.append(('Others', 'label_paths.csv', 'All label paths with document counts'))

    o.other('held_out_categories.json', ev['held_out_categories'],
            'Level-2 categories withheld for the unseen-category experiment')

    findings = f"""KEY FINDINGS (all measured, {len(seeds)} seeds, product-disjoint split)

1. Consistency-constrained decoding works.
   Hierarchical F1  {_pm(R,'Proposed (-consistency)','hF1')}  ->  {_pm(R,'Proposed (full)','hF1')}
   Invalid paths    {_pm(R,'Proposed (-consistency)','invalid_path_rate')}%  ->  {_pm(R,'Proposed (full)','invalid_path_rate')}%
   Paired t-test against the strongest baseline: p = {stats.ttest_rel(base,_stat(R,'Top-down classifier chain (predefined tree)','hF1'))[1]:.1e}

2. Hierarchy discovery contributes nothing.
   Removing it changes hierarchical F1 by {base.mean()-_stat(R,'Proposed (-discovery)','hF1').mean():+.2f} points
   (p = {stats.ttest_rel(base,_stat(R,'Proposed (-discovery)','hF1'))[1]:.2f}).
   Agreement with the reference hierarchy: ARI {hr['Cat1']['ARI']:.3f} / {hr['Cat2']['ARI']:.3f} / {hr['Cat3']['ARI']:.3f}
   (0 = chance).

3. Dynamic cluster induction cannot detect unseen categories.
   Detection at the specified threshold: {ev['detection_rate_new']*100:.1f}%.
   Threshold-free ROC AUC: {ev['roc_auc']:.3f} (chance = 0.500).

4. Conceptual consequence.
   The "evolving label graph" attains precision and recall of 1.000 against the
   reference edge set only because the edges are enumerated from the training
   annotations. That is supervised structure extraction, not discovery. The
   measured gain in (1) comes from this extracted lattice.

5. True task difficulty.
   Measured accuracy {_pm(R,'Proposed (full)','acc_L1')} / {_pm(R,'Proposed (full)','acc_L2')} / {_pm(R,'Proposed (full)','acc_L3')}
   against the 92.4 / 88.6 / 84.2 stated in the submitted manuscript.
   Macro-F1 at level 3 is {_pm(R,'Proposed (full)','macroF1_L3')}: the long tail dominates.
"""
    o.other('key_findings.txt', findings, 'Narrative summary of the measured findings')

    man = pd.DataFrame(o.manifest, columns=['Folder', 'File', 'Description'])
    man.to_csv(os.path.join(root, 'Others', 'manifest.csv'), index=False)
    with open(os.path.join(root, 'README.md'), 'w') as fh:
        fh.write('# Experimental outputs\n\n'
                 f'Generated {datetime.datetime.now():%Y-%m-%d %H:%M}. '
                 'Every figure and table is derived from the raw JSON in `Others/`; '
                 'no value is transcribed by hand.\n\n')
        fh.write(man.to_markdown(index=False) + '\n')
    print(f'\n{len(o.manifest)+1} artefacts written under {root}/', flush=True)
    return man


## Experiment runners

In [ ]:
%%writefile run_baselines.py
import time, json, pandas as pd, numpy as np
import htc_data as H, baselines as B, evaluate as E

df = pd.read_pickle('labelled.pkl')
e2, e3 = H.valid_edge_sets(df)
res = {}
for proto, sp in [('P1_published', H.split_published(df)),
                  ('P2_product_disjoint', H.split_product_disjoint(df, seed=0))]:
    tr = df[sp == 'train']; te = df[sp == 'test']
    vec = B.vectoriser(); Xtr = vec.fit_transform(tr.Text); Xte = vec.transform(te.Text)
    print(f'\n=== {proto}  train={len(tr)} test={len(te)} feats={Xtr.shape[1]}', flush=True)
    truth = list(zip(te.Cat1, te.Cat2, te.Cat3))
    for M in [B.FlatLeaf, B.PerLevel, B.TopDownTree]:
        t0 = time.time(); m = M(seed=0).fit(Xtr, tr); pred = m.predict(Xte)
        s = E.score(truth, pred, e2, e3); s['fit_s'] = round(time.time()-t0, 1)
        res[f'{proto}|{M.name}'] = s
        print(f'{M.name:45s} {E.fmt(s)}  [{s["fit_s"]}s]', flush=True)
json.dump(res, open('results_baselines.json','w'), indent=1)


In [ ]:
%%writefile run_proposed.py
import time, json, pandas as pd, numpy as np
import htc_data as H, baselines as B, evaluate as E, proposed as PR, repr_util as REP

df = pd.read_pickle('labelled.pkl'); e2, e3 = H.valid_edge_sets(df)
sp = H.split_product_disjoint(df, seed=0)
tr, va, te = df[sp == 'train'], df[sp == 'val'], df[sp == 'test']
vec = B.vectoriser()
Xtr = vec.fit_transform(tr.Text); Xva = vec.transform(va.Text); Xte = vec.transform(te.Text)
t0 = time.time()
Ztr, (Zva, Zte) = REP.build(Xtr, [Xva, Xte], tr, [va, te], 0)
print(f'representation {REP.describe()} [{time.time()-t0:.0f}s]', flush=True)
truth = list(zip(te.Cat1, te.Cat2, te.Cat3))
res = {}
for tag, kw in [('full', {}),
                ('-consistency', {'use_consistency': False}),
                ('-discovery', {'use_clusters': False}),
                ('-calibration', {'use_calibration': False})]:
    t0 = time.time()
    m = PR.HierarchyAgnostic(seed=0, n_clusters=300, **kw).fit(Ztr, tr, Zva, va, Xtr, Xva)
    s = E.score(truth, m.predict(Zte, Xte), e2, e3)
    s['fit_s'] = round(time.time() - t0, 1); s['T'] = m.T
    res[tag] = s
    print(f'{tag:15s} {E.fmt(s)} [{s["fit_s"]}s] T={m.T}', flush=True)
json.dump(res, open('results_proposed.json', 'w'), indent=1)
print('DONE', flush=True)


In [ ]:
%%writefile run_seeds.py
"""Multi-seed evaluation. Seed controls the product-disjoint partition, the SVD,
the clustering and the solver, so the reported spread covers both data-split and
optimisation variability."""
import os, time, json, sys, pandas as pd, numpy as np
import htc_data as H, baselines as B, evaluate as E, proposed as PR, repr_util as REP

SEEDS = [int(x) for x in os.environ.get('HTC_SEEDS', '0,1,2,3,4').split(',')]
df = pd.read_pickle('labelled.pkl'); e2, e3 = H.valid_edge_sets(df)
out = {}
for seed in SEEDS:
    sp = H.split_product_disjoint(df, seed=seed)
    tr, va, te = df[sp == 'train'], df[sp == 'val'], df[sp == 'test']
    vec = B.vectoriser()
    Xtr = vec.fit_transform(tr.Text); Xva = vec.transform(va.Text); Xte = vec.transform(te.Text)
    Ztr, (Zva, Zte) = REP.build(Xtr, [Xva, Xte], tr, [va, te], seed)
    truth = list(zip(te.Cat1, te.Cat2, te.Cat3))
    print(f'--- seed {seed}  train={len(tr)} val={len(va)} test={len(te)}', flush=True)
    for M in [B.FlatLeaf, B.PerLevel, B.TopDownTree]:
        t0 = time.time(); pred = M(seed=seed).fit(Xtr, tr).predict(Xte)
        s = E.score(truth, pred, e2, e3)
        out.setdefault(M.name, {})[seed] = s
        print(f'  {M.name:44s} {E.fmt(s)} [{time.time()-t0:.0f}s]', flush=True)
    for tag, kw in [('Proposed (full)', {}),
                    ('Proposed (-consistency)', {'use_consistency': False}),
                    ('Proposed (-discovery)', {'use_clusters': False}),
                    ('Proposed (-calibration)', {'use_calibration': False})]:
        t0 = time.time()
        m = PR.HierarchyAgnostic(seed=seed, n_clusters=300, **kw).fit(Ztr, tr, Zva, va, Xtr, Xva)
        s = E.score(truth, m.predict(Zte, Xte), e2, e3)
        out.setdefault(tag, {})[seed] = s
        print(f'  {tag:44s} {E.fmt(s)} [{time.time()-t0:.0f}s]', flush=True)
    json.dump(out, open('results_seeds.json', 'w'), indent=1)
print('DONE', flush=True)


In [ ]:
%%writefile run_discovery.py
"""
Two experiments the manuscript claims but never performs.

E1  Hierarchy-recovery quality
    Does the discovered structure correspond to the reference hierarchy at all?
    Reported as adjusted Rand index and normalised mutual information between
    the discovered partition and each reference level, at matched granularity
    (K = 6, 64, 464), plus the precision/recall of the discovered parent-child
    edge set against the reference DAG.

E2  Evolving label space / unseen categories
    Whole level-2 categories are withheld from training. At test time the
    dynamic-cluster-induction rule (Eq. 10) must flag their documents as
    out-of-structure. Reported as detection rate at the operating threshold
    tau = mu - 2 sigma, and as ROC AUC of the nearest-centroid similarity,
    which is threshold-free.
"""
import json, time
import numpy as np, pandas as pd
from sklearn.cluster import MiniBatchKMeans, AgglomerativeClustering
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             roc_auc_score)
import htc_data as H, baselines as B, proposed as PR, repr_util as REP

df = pd.read_pickle('labelled.pkl')
sp = H.split_product_disjoint(df, seed=0)
tr, te = df[sp == 'train'], df[sp == 'test']
vec = B.vectoriser()
Xtr = vec.fit_transform(tr.Text); Xte = vec.transform(te.Text)
Ztr, (Zte,) = REP.build(Xtr, [Xte], tr, [te], 0)
res = {}

# ---------------------------------------------------------------- E1
print('E1 hierarchy recovery', flush=True)
e1 = {}
for K, lv in [(6, 'Cat1'), (64, 'Cat2'), (464, 'Cat3')]:
    km = MiniBatchKMeans(n_clusters=K, random_state=0, n_init=5,
                         batch_size=2048, max_iter=200).fit(Ztr)
    a = km.labels_
    y = tr[lv].values
    e1[lv] = {'K': K,
              'ARI': float(adjusted_rand_score(y, a)),
              'NMI': float(normalized_mutual_info_score(y, a))}
    print('  ', lv, e1[lv], flush=True)
res['hierarchy_recovery'] = e1

# nested structure: cluster at K=64, agglomerate centroids to 6, compare to Cat1
km64 = MiniBatchKMeans(n_clusters=64, random_state=0, n_init=5,
                       batch_size=2048, max_iter=200).fit(Ztr)
agg = AgglomerativeClustering(n_clusters=6, linkage='ward').fit(km64.cluster_centers_)
coarse = agg.labels_[km64.labels_]
res['nested_coarse_vs_Cat1'] = {
    'ARI': float(adjusted_rand_score(tr.Cat1.values, coarse)),
    'NMI': float(normalized_mutual_info_score(tr.Cat1.values, coarse))}
print('  nested 64->6 vs Cat1', res['nested_coarse_vs_Cat1'], flush=True)

# discovered edge set vs reference
m = PR.HierarchyAgnostic(seed=0, n_clusters=300).__class__(seed=0, n_clusters=300)
km300 = MiniBatchKMeans(n_clusters=300, random_state=0, n_init=5,
                        batch_size=2048, max_iter=200).fit(Ztr)
m.km = km300
m._label_graph(km300.labels_, tr)
ref2, ref3 = H.valid_edge_sets(tr)
def pr(disc, ref):
    tp = len(disc & ref)
    return {'n_discovered': len(disc), 'n_reference': len(ref),
            'precision': tp / max(len(disc), 1), 'recall': tp / max(len(ref), 1)}
res['edges_L1_L2'] = pr(m.edges2, ref2)
res['edges_L2_L3'] = pr(m.edges3, ref3)
print('  edges', res['edges_L1_L2'], res['edges_L2_L3'], flush=True)

# ---------------------------------------------------------------- E2
print('E2 evolving label space', flush=True)
rng = np.random.RandomState(0)
cats2 = sorted(tr.Cat2.unique())
held = list(rng.choice(cats2, size=8, replace=False))
tr_seen = tr[~tr.Cat2.isin(held)]
Xs = vec.fit_transform(tr_seen.Text)
ues = PR.UnifiedEmbedding(256, 0).fit(Xs)
Zs = ues.transform(Xs)
km = MiniBatchKMeans(n_clusters=300, random_state=0, n_init=5,
                     batch_size=2048, max_iter=200).fit(Zs)
sim_tr = (Zs * km.cluster_centers_[km.labels_]).sum(1)
tau = float(sim_tr.mean() - 2 * sim_tr.std())
Zt = ues.transform(vec.transform(te.Text))
sim_te = (Zt @ km.cluster_centers_.T).max(1)
is_new = te.Cat2.isin(held).values
res['evolving'] = {
    'held_out_categories': held,
    'n_test_new': int(is_new.sum()), 'n_test_seen': int((~is_new).sum()),
    'tau': tau,
    'detection_rate_new': float((sim_te[is_new] < tau).mean()),
    'false_alarm_rate_seen': float((sim_te[~is_new] < tau).mean()),
    'roc_auc': float(roc_auc_score(is_new, -sim_te)),
    'mean_sim_new': float(sim_te[is_new].mean()),
    'mean_sim_seen': float(sim_te[~is_new].mean())}
print('  ', res['evolving'], flush=True)

json.dump(res, open('results_discovery.json', 'w'), indent=1)
print('DONE', flush=True)


## Stage 1 — corpus and structure

In [ ]:
# ------------------------------------------------------------------ Stage 1
# Load the labelled corpus and describe its structure.
import pandas as pd, htc_data as H
df = H.load_labelled()
df.to_pickle('labelled.pkl')
ref = H.reference_hierarchy(df)
print(f'documents after cleaning : {len(df):,}')
for c in H.LEVELS:
    print(f'{c} categories            : {df[c].nunique()}')
print(f'distinct label paths      : {df.groupby(H.LEVELS).ngroups}')
print(f"level-2 with >1 parent    : {ref['cat2_multi_parent']}")
print(f"level-3 with >1 parent    : {ref['cat3_multi_parent']}  <- the reference structure is a DAG, not a tree")
sp = H.split_product_disjoint(df, seed=0)
print('\nproduct-disjoint split    :', sp.value_counts().to_dict())
print('product overlap train/test:',
      len(set(df[sp=='train'].productId) & set(df[sp=='test'].productId)))
df.head(3)

## Stage 2 — baselines

In [ ]:
# ------------------------------------------------------------------ Stage 2
# Baselines under both protocols. Establishes the effect of the leaky split.
%run run_baselines.py

## Stage 2b — transformer representation (optional)

Runs only when `ENCODER == 'transformer'`. Embeddings are computed once for the
whole labelled corpus and cached to Drive, so later stages and re-runs do not
recompute them.

In [ ]:
#@title Compute and cache transformer embeddings
import numpy as np, pandas as pd, os
CACHE = os.path.join(OUT_ROOT, 'Others', f'embeddings_{MODEL_NAME.replace("/","_")}.npy')
if ENCODER == 'transformer':
    import embeddings as EMB
    df = pd.read_pickle('labelled.pkl')
    if os.path.exists(CACHE):
        Zall = np.load(CACHE)
        print('loaded cached embeddings', Zall.shape)
    else:
        enc = EMB.TransformerEncoder(MODEL_NAME, max_length=MAX_LENGTH, batch_size=64)
        Zall = enc.transform(texts=df.Text.tolist())
        np.save(CACHE, Zall)
        print('computed and cached', Zall.shape, '->', CACHE)
    os.environ['HTC_EMB_CACHE'] = CACHE   # every later stage now uses these
    print('later stages will use the transformer representation')
else:
    print('ENCODER = tfidf_svd; skipping (set ENCODER = "transformer" to enable)')

## Stage 3 — proposed framework and ablations

In [ ]:
# ------------------------------------------------------------------ Stage 3
# The proposed framework and its ablations, single seed (fast sanity pass).
%run run_proposed.py

## Stage 4 — multi-seed evaluation

In [ ]:
# ------------------------------------------------------------------ Stage 4
# Multi-seed evaluation. This is the long stage: SEEDS x 7 model fits.
# Set SEEDS = [0] in run_seeds.py for a quick pass.
%run run_seeds.py

## Stage 5 — does it discover a hierarchy?

In [ ]:
# ------------------------------------------------------------------ Stage 5
# The two experiments the manuscript claims but never performed:
#   E1 does the discovered structure match the reference hierarchy?
#   E2 can dynamic cluster induction detect categories held out of training?
%run run_discovery.py

## Stage 6 — assemble the output folders in Drive

In [ ]:
# ------------------------------------------------------------------ Stage 6
# Build Figures/, Tables/ and Others/ from the measured JSON.
import outputs, pandas as pd
manifest = outputs.build(OUT_ROOT, results_dir='.', df=pd.read_pickle('labelled.pkl'))
manifest

## Verification

In [ ]:
# ------------------------------------------------------------------ Verify
# Independent check: re-read the written tables and confirm the headline
# numbers agree with the raw per-seed JSON they were derived from.
import json, numpy as np, pandas as pd, os
raw = json.load(open(os.path.join(OUT_ROOT, 'Others', 'raw_results_per_seed.json')))
tab = pd.read_csv(os.path.join(OUT_ROOT, 'Tables', 'table2_main_results.csv'))
v = np.array([raw['Proposed (full)'][s]['hF1'] for s in sorted(raw['Proposed (full)'], key=int)]) * 100
stated = tab.loc[tab.Model == 'Proposed (full)', 'Hier. F1'].iloc[0]
recomputed = f'{v.mean():.2f} ± {v.std(ddof=1):.2f}'
print('table states :', stated)
print('recomputed   :', recomputed)
assert stated == recomputed, 'table and raw results disagree'
print('\nOK - tables agree with the raw measurements')

In [ ]:
#@title Confirm what landed in Drive
import os
for sub in ('Figures', 'Tables', 'Others'):
    d = os.path.join(OUT_ROOT, sub)
    files = sorted(os.listdir(d))
    print(f'\n{sub}/  ({len(files)} files)')
    for f in files:
        print('   ', f, f'{os.path.getsize(os.path.join(d, f))//1024} KB')